# TR-TRY: Translation Retrieval for Akkadian

TF-IDF-based translation retrieval from training corpus.
No GPU required - runs on CPU only.

**Approach:**
1. Build TF-IDF index (char + word n-grams) from training transliterations
2. For each test sample, retrieve the most similar training pair
3. Use the training translation as the prediction
4. Apply postprocessing (bracket removal, dedup, cleanup)

**Local GeoMean: 40.52**

In [ ]:
import os, re, unicodedata, math
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Debug: check what's available
print("=== /kaggle/input/ contents ===")
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    for item in sorted(os.listdir(input_dir)):
        full = os.path.join(input_dir, item)
        if os.path.isdir(full):
            files = os.listdir(full)
            print(f"  {item}/ ({len(files)} files)")
            for f in sorted(files)[:5]:
                fpath = os.path.join(full, f)
                sz = os.path.getsize(fpath) if os.path.isfile(fpath) else "dir"
                print(f"    {f} ({sz})")
        else:
            print(f"  {item}")
else:
    print("  /kaggle/input does NOT exist!")
print("=" * 40)

## Configuration

In [ ]:
import os

# Competition data path - check both standard and competitions/ mount points
DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

# Retriever parameters
W_CHAR = 0.75
W_WORD = 0.20
W_SEQ = 0.05
TOP_K = 60
RERANK_K = 10
LEN_PENALTY_POWER = 0.3
MIN_ACCEPT_SCORE = 0.10

## Normalization

In [ ]:
def normalize_akkadian(text):
    """Normalize Akkadian transliteration for retrieval matching."""
    if pd.isna(text) or not isinstance(text, str):
        return ""

    x = str(text).lower().strip()

    # Remove subscript/superscript digits for homophone folding
    x = re.sub(r"[\u2080-\u2089\u2070-\u2079\u00b9\u00b2\u00b30-9]", "", x)

    # Normalize determinatives
    x = re.sub(r"\{([^}]+)\}", r" DET_\1 ", x)
    x = re.sub(r"\(([a-z]{1,6})\)", r" DET_\1 ", x)

    # Remove editorial brackets but keep content
    for ch in "[]<>":
        x = x.replace(ch, " ")

    # Remove apostrophe-like signs
    x = re.sub(r"[\u02be\u02bf\u02c0\u02c1']", "", x)

    # Gap markers
    x = x.replace("\u2026", " GAP ").replace("...", " GAP ")
    x = re.sub(r"\b[xX]{1,3}\b", " GAP ", x)

    # Dot and plus separators
    x = x.replace(".", " ").replace("+", "")

    # Unicode NFKD normalization (remove diacritics)
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))

    # Keep only ASCII alphanumeric, hyphens, underscores, spaces
    x = re.sub(r"[^a-z0-9\-_ ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

## Postprocessing

In [ ]:
_SUBSCRIPT_TABLE = str.maketrans("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089", "0123456789")
_BAD_OUTPUT_CHARS = '!?()\"\u2014\u2013<>\u2308\u230b\u230a[]+\u02be/;'
_BRACKET_PATTERN = re.compile(r"[\(\[][^\)\]]*[\)\]]")

_FRACTION_MAP = {
    "1/2": "\u00bd", "1/3": "\u2153", "2/3": "\u2154",
    "1/4": "\u00bc", "3/4": "\u00be", "1/5": "\u2155",
    "1/6": "\u2159", "5/6": "\u215a", "1/8": "\u215b",
}

_SHORT_INPUT_MAP = {
    "a-na": "To", "um-ma": "saying:", "IGI": "Witnesses:",
    "KI\u0160IB": "Seal of", "ma-na": "mina of silver",
    "\u0161u-ma": "If he does not pay", "i-na": "In",
    "ITU.KAM": "Month:", "i\u0161-t\u00f9": "From the",
    "K\u00d9.BABBAR": "silver", "G\u00cdN": "shekels of silver",
    "\u00fa-\u1e63a-\u00e1b": "he will add interest",
    "li-mu-um": "Eponymy of", "\u00f9": "Also,",
    "\u0161a": "of", "l\u00e1": "not", "URUDU": "copper",
    "x": "broken", "\u2026": "broken", "a-ma-kam": "Here",
    "en-um-a-\u0161ur": "Ennum-Assur",
    "k\u00e0-ru-um": "The colony",
}

# Common Akkadian diacritic -> ASCII mappings
_DIACRITIC_MAP = {
    '\u0161': 's', '\u0160': 'S',  # š -> s, Š -> S
    '\u1e63': 's', '\u1e62': 'S',  # ṣ -> s, Ṣ -> S
    '\u1e6d': 't', '\u1e6c': 'T',  # ṭ -> t, Ṭ -> T
    '\u1e2b': 'h', '\u1e2a': 'H',  # ḫ -> h, Ḫ -> H
    '\u0101': 'a', '\u0100': 'A',  # ā -> a, Ā -> A
    '\u0113': 'e', '\u0112': 'E',  # ē -> e, Ē -> E
    '\u012b': 'i', '\u012a': 'I',  # ī -> i, Ī -> I
    '\u016b': 'u', '\u016a': 'U',  # ū -> u, Ū -> U
    '\u00e0': 'a', '\u00e1': 'a', '\u00e8': 'e', '\u00e9': 'e',
    '\u00ec': 'i', '\u00ed': 'i', '\u00f2': 'o', '\u00f3': 'o',
    '\u00f9': 'u', '\u00fa': 'u',
}


def sanitize_ascii(text):
    """Replace non-ASCII characters with ASCII equivalents."""
    if not text:
        return text
    # Apply known diacritic mappings first
    for src, dst in _DIACRITIC_MAP.items():
        text = text.replace(src, dst)
    # NFKD decompose remaining non-ASCII and strip combining marks
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    # Replace any remaining non-ASCII with closest ASCII
    result = []
    for ch in text:
        if ord(ch) < 128:
            result.append(ch)
        else:
            result.append(' ')
    return ''.join(result)


def postprocess(text):
    """Clean retrieved translation for submission."""
    if not isinstance(text, str) or not text.strip():
        return ""

    t = text
    # Transliterate special chars
    t = t.replace("\u1e2b", "h").replace("\u1e2a", "H")
    t = t.translate(_SUBSCRIPT_TABLE)

    # Remove bracket content (improves LB)
    t = t.replace("<gap>", "\x00GAP\x00").replace("<big_gap>", "\x00BIG\x00")
    t = _BRACKET_PATTERN.sub("", t)
    t = t.replace("\x00GAP\x00", "").replace("\x00BIG\x00", "")

    # Remove special chars
    t = t.translate(str.maketrans("", "", _BAD_OUTPUT_CHARS))

    # Convert fractions
    for frac, symbol in _FRACTION_MAP.items():
        t = t.replace(frac, symbol)

    # Remove repeated words/phrases
    t = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", t)
    for n in range(4, 1, -1):
        pat = r"\b((?:\w+\s+){" + str(n - 1) + r"}\w+)(?:\s+\1\b)+"
        t = re.sub(pat, r"\1", t)

    t = re.sub(r"\s+", " ", t).strip().strip("-").strip()

    # Sanitize to ASCII-safe output
    t = sanitize_ascii(t)
    t = re.sub(r"\s+", " ", t).strip()

    return t


def get_short_translation(raw_input):
    """Return dictionary translation for 1-token inputs."""
    tokens = str(raw_input).strip().split()
    if len(tokens) == 1:
        return _SHORT_INPUT_MAP.get(tokens[0])
    return None

## Translation Retriever

In [ ]:
class TranslationRetriever:
    """TF-IDF-based translation retrieval from training corpus."""

    def __init__(self, train_path):
        df = pd.read_csv(train_path)
        df = df.dropna(subset=["transliteration", "translation"])

        self.src_raw = df["transliteration"].tolist()
        self.tgt_raw = df["translation"].tolist()
        self.src_norm = [normalize_akkadian(s) for s in self.src_raw]

        # Char n-gram TF-IDF (3-6 grams)
        self.char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6))
        self.X_char = self.char_vec.fit_transform(self.src_norm)

        # Word n-gram TF-IDF (1-2 grams)
        self.word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
        self.X_word = self.word_vec.fit_transform(self.src_norm)

        # Source lengths for length penalty
        self.src_lengths = np.array(
            [len(s.split()) for s in self.src_norm], dtype=np.float32
        )

        print(f"Indexed {len(self.src_raw)} training pairs")
        print(f"Char vocab: {len(self.char_vec.vocabulary_)}, Word vocab: {len(self.word_vec.vocabulary_)}")

    def retrieve(self, query):
        """Retrieve best matching translation. Returns (translation, score)."""
        q_norm = normalize_akkadian(query)
        if not q_norm:
            return "", 0.0

        # TF-IDF similarity
        q_char = self.char_vec.transform([q_norm])
        q_word = self.word_vec.transform([q_norm])

        sc = (q_char @ self.X_char.T).toarray()[0]
        sw = (q_word @ self.X_word.T).toarray()[0]

        combined = W_CHAR * sc + W_WORD * sw

        # Top-K with length penalty
        k = min(TOP_K, len(self.src_norm))
        cand_idx = np.argpartition(-combined, k)[:k]

        q_len = max(1, len(q_norm.split()))
        ratio = self.src_lengths[cand_idx] / q_len
        lps = np.exp(-np.abs(np.log(ratio + 1e-5)) * LEN_PENALTY_POWER)
        final_scores = combined[cand_idx] * lps

        # Rerank top candidates with SequenceMatcher
        rerank_idx = cand_idx[np.argsort(-final_scores)[:RERANK_K]]

        best_score = -1.0
        best_idx = -1

        for idx in rerank_idx:
            seq_score = SequenceMatcher(None, q_norm, self.src_norm[idx]).ratio()
            total = final_scores[np.where(cand_idx == idx)[0][0]] + W_SEQ * seq_score

            if total > best_score:
                best_score = total
                best_idx = idx

        if best_score > MIN_ACCEPT_SCORE and best_idx >= 0:
            return self.tgt_raw[best_idx], best_score
        return "", 0.0

## Build Index

In [ ]:
import time

start = time.time()
retriever = TranslationRetriever(TRAIN_PATH)
print(f"Index built in {time.time() - start:.1f}s")

## Inference

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test samples: {len(test_df)}")

results = []
scores = []

for _, row in test_df.iterrows():
    raw_text = str(row["transliteration"]) if pd.notna(row.get("transliteration", None)) else ""

    # Short input lookup
    short = get_short_translation(raw_text)
    if short is not None:
        results.append({"id": row["id"], "translation": short})
        scores.append(1.0)
        continue

    # Retrieve from training corpus
    translation, score = retriever.retrieve(raw_text)

    if translation:
        translation = postprocess(translation)

    results.append({"id": row["id"], "translation": translation})
    scores.append(score)

sub_df = pd.DataFrame(results)
print(f"\nSubmission shape: {sub_df.shape}")
print(f"Avg retrieval score: {np.mean(scores):.4f}")
print(f"Min score: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
print(f"Empty translations: {(sub_df['translation'] == '').sum()}")
sub_df.head(10)

## Save Submission

In [ ]:
# Ensure no empty/NaN translations
sub_df['translation'] = sub_df['translation'].fillna('broken text')
sub_df['translation'] = sub_df['translation'].apply(lambda x: x if x.strip() else 'broken text')

# Ensure id is integer
sub_df['id'] = sub_df['id'].astype(int)

# Validate before saving
print("=== Submission Validation ===")
print(f"Shape: {sub_df.shape}")
print(f"Columns: {list(sub_df.columns)}")
print(f"ID dtype: {sub_df['id'].dtype}")
print(f"NaN count: {sub_df.isnull().sum().to_dict()}")
print(f"Empty translations: {(sub_df['translation'] == '').sum()}")
non_ascii_count = sum(1 for t in sub_df['translation'] if any(ord(c) > 127 for c in str(t)))
print(f"Rows with non-ASCII: {non_ascii_count}")

# Save with explicit encoding
sub_df.to_csv("submission.csv", index=False, encoding='utf-8')

# Verify saved file
import csv
with open("submission.csv", 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    rows = list(reader)
    print(f"\nCSV rows (incl header): {len(rows)}")
    print(f"Header: {rows[0]}")
    for row in rows[1:]:
        print(f"  ID {row[0]}: len={len(row[1])}, ascii_only={all(ord(c) < 128 for c in row[1])}")

print("\nSaved submission.csv")
print(f"\nSample outputs:")
for _, row in sub_df.iterrows():
    print(f"  ID {row['id']}: {str(row['translation'])[:120]}...")